# Benchmark: Scalability vs Grid Size (Barrier Method)

Same sweep as the SLSQP scalability benchmark but using the penalty -> log-barrier
L-BFGS-B solver. Compares runtime, L2 error, and convergence across grid sizes.

Sizes tested: **5x5, 8x8, 10x10, 12x12, 15x15, 18x18, 20x20**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
)
from dvfopt.core.iterative2d_barrier import iterative_2d_barrier
from benchmark_utils import (
    run_correction,
    plot_jac_heatmaps,
    plot_correction_magnitude,
    plot_jdet_histograms,
)
from benchmark_utils import get_output_dir, save_figure, save_results_csv, save_summary_json, log_run_header, log_run_footer, results_to_rows, show_and_save, reset_figure_counter


In [ ]:
METHOD = "barrier"
NOTEBOOK_NAME = "scalability"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)


## Configuration

In [ ]:
GRID_SIZES = [5, 8, 10, 12, 15, 18, 20]
BASE_SHAPE = (3, 1, 5, 5)
MAX_MAGNITUDE = 5.0
SEED = 42

## Generate test fields

In [ ]:
base_dvf = generate_random_dvf(BASE_SHAPE, MAX_MAGNITUDE, SEED)
test_fields = {}
for size in GRID_SIZES:
    dvf = scale_dvf(base_dvf, (size, size))
    phi = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac = jacobian_det2D(phi)
    n_neg = int((jac <= 0).sum())
    test_fields[size] = dvf
    print(f"  {size:>4d}x{size:<4d}  neg-Jdet: {n_neg:>5d}  "
          f"min Jdet: {jac.min():+.4f}  total pixels: {size*size}")

## Run barrier corrections

In [ ]:
solver = iterative_2d_barrier
solver_name = "barrier (L-BFGS-B)"
print(f"Solver: {solver_name}
")

results = {}
print(f"{"Size":>10s}  {"Time (s)":>10s}  {"Neg init":>10s}  {"Neg final":>10s}  "
      f"{"Min Jdet":>10s}  {"L2 Error":>10s}")
print("-" * 70)

for size in GRID_SIZES:
    dvf = test_fields[size]
    results[size] = run_correction(dvf, solver)
    r = results[size]
    print(f"{size:>4d}x{size:<4d}  {r["time"]:>10.2f}  {r["n_neg_init"]:>10d}  "
          f"{r["n_neg_final"]:>10d}  {r["min_jdet"]:>+10.6f}  {r["l2_err"]:>10.4f}")

## Jacobian heatmaps

In [ ]:
plot_jac_heatmaps(
    [[results[s]["jac_init"][0] for s in GRID_SIZES],
     [results[s]["jac_final"][0] for s in GRID_SIZES]],
    [f"{s}x{s}" for s in GRID_SIZES],
    title="Jacobian Determinant - Before vs After (Barrier)",
)
show_and_save(OUTPUT_DIR)


## Correction magnitude

In [ ]:
plot_correction_magnitude(
    [(results[s]["phi"], results[s]["phi_init"]) for s in GRID_SIZES],
    [f"{s}x{s}" for s in GRID_SIZES],
    title="Per-pixel displacement change (barrier)",
)
show_and_save(OUTPUT_DIR)


## Jdet distributions

In [ ]:
plot_jdet_histograms(
    [[("Before", results[s]["jac_init"][0]),
      ("After", results[s]["jac_final"][0])] for s in GRID_SIZES],
    [f"{s}x{s}" for s in GRID_SIZES],
    title="Jacobian Determinant Distribution (Barrier)",
)
show_and_save(OUTPUT_DIR)


## Scaling plots

In [ ]:
sizes = list(results.keys())
times = [results[s]["time"] for s in sizes]
l2s = [results[s]["l2_err"] for s in sizes]
negs = [results[s]["n_neg_init"] for s in sizes]
pixels = [s * s for s in sizes]
pct_neg = [100 * results[s]["n_neg_init"] / (s * s) for s in sizes]
min_jdet_init = [results[s]["min_jdet_init"] for s in sizes]
min_jdet_final = [results[s]["min_jdet"] for s in sizes]

fig, axes = plt.subplots(3, 2, figsize=(12, 13))

ax = axes[0, 0]
ax.plot(sizes, times, "o-", color="tab:blue")
ax.set_xlabel("Grid size (NxN)"); ax.set_ylabel("Time (s)")
ax.set_title("Runtime vs Grid Size"); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.loglog(pixels, times, "o-", color="tab:blue")
log_p = np.log(pixels); log_t = np.log(times)
alpha, log_c = np.polyfit(log_p, log_t, 1)
fit_t = np.exp(log_c) * np.array(pixels) ** alpha
ax.loglog(pixels, fit_t, "--", color="tab:red", label=f"fit: O(n^{alpha:.2f})")
ax.set_xlabel("Total pixels"); ax.set_ylabel("Time (s)")
ax.set_title("Runtime Scaling (log-log)"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(sizes, l2s, "s-", color="tab:orange")
ax.set_xlabel("Grid size (NxN)"); ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation vs Grid Size"); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(sizes, negs, "^-", color="tab:green", label="Count")
ax.set_xlabel("Grid size (NxN)"); ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding Pixels vs Grid Size"); ax.grid(True, alpha=0.3)

ax = axes[2, 0]
x = np.arange(len(sizes)); w = 0.35
ax.bar(x - w/2, min_jdet_init, w, label="Before", color="tab:red", alpha=0.7)
ax.bar(x + w/2, min_jdet_final, w, label="After", color="tab:blue", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels([f"{s}x{s}" for s in sizes], rotation=30)
ax.set_ylabel("Min Jacobian determinant"); ax.set_title("Worst-case Jdet"); ax.legend()

ax = axes[2, 1]
time_per_neg = [results[s]["time"]/max(results[s]["n_neg_init"],1) for s in sizes]
ax.bar([f"{s}x{s}" for s in sizes], time_per_neg, color="tab:purple")
ax.set_ylabel("Time per neg-Jdet pixel (s)"); ax.set_title("Cost per Folding Pixel")
plt.sca(ax); plt.xticks(rotation=30)

plt.suptitle(f"Scalability - barrier solver", fontsize=14)
plt.tight_layout(); show_and_save(OUTPUT_DIR)

In [ ]:
# --- Save results ---
if 'results' in dir() and isinstance(results, dict) and results:
    rows, cols = results_to_rows(results)
    save_results_csv(rows, cols, OUTPUT_DIR)
    summary = log_run_footer(summary, results)
    save_summary_json(summary, OUTPUT_DIR)
elif 'mag_results' in dir():
    rows, cols = results_to_rows(mag_results)
    save_results_csv(rows, cols, OUTPUT_DIR, name='results_magnitude')
    if 'density_results' in dir():
        rows2, cols2 = results_to_rows(density_results)
        save_results_csv(rows2, cols2, OUTPUT_DIR, name='results_density')
    combined = {**mag_results, **density_results} if 'density_results' in dir() else mag_results
    summary = log_run_footer(summary, combined)
    save_summary_json(summary, OUTPUT_DIR)
else:
    save_summary_json(summary, OUTPUT_DIR)
    print('No results dict found; saved summary only.')
